In [19]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import matplotlib as mpl
import plotconfig
from matplotlib.lines import Line2D

In [ ]:
bbr_version = "bbr3"

args = {
    "chaos": "on_1",
    "timestamp": ["20251119", "2025120", "2025112", "2025112"],
    "cca": ["cubic", bbr_version],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel6-13-BBRv3"],

    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [100],
    "delay_rtt": [10,20,30,40],
    "deadline_run": [1000000, 10000000, 20000000],
}

metric = "bits_per_second"

baselogpath = "../data"

In [21]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 8790
end 8790
bytes 8790
bits_per_second 8790
mbps_timeseries 8790
rttms_timeseries 8790
retransmits 8790
timestamp 8790
iteration 8790
cpu_host_total 8790
cpu_host_user 8790
cpu_host_system 8790
cpu_remote_total 8790
chaos 8790
deadline_run 8790
deadline_period 8790
os 8790
bdp 8790
setup 8790
cca 8790
cpus 8790
kernel 8790
mode 8790
loss 8790
rate 8790
delay_rtt 8790
buffer_size_bytes 8790
parallel 8790
socket_buffer 8790
app_buffer 8790
n 8790
sysctl_cmd 8790
vm 8790
bandwidth_delay_product 8790
loss_mode 8790
vms 8790
pacing 8790
hyperthreading 8790
tso 8790
qdisc 8790
hpet 8790
tsc 8790
hostq 8790
loadperc 8790
deadline_period_factor 8790
random_loss_rate 8790
gemodel_q 8790
original_cca 8790
test_cca 8790
default_qdisc 8790
json 8790


In [22]:
df["slice_perc"] = round((df["deadline_run"]/df["deadline_period"])*100)
df["mbps"] = df["bits_per_second"]/1000000

In [23]:
def strip(df, savefig = False):    
    if savefig:
        mpl.use('agg')
    plotconfig.configure_conext()
    width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
    height = width*(1/3)
    FIG_SIZE = (width, height)
    
    paletti = plotconfig.COLORS[:5]
    fig, axs = plt.subplots(1,3,figsize=FIG_SIZE, sharey=True,constrained_layout=True)


    for it, runti in enumerate(sorted(df["deadline_run"].unique())):
        data = df[df["deadline_run"] == runti]
        data = data.sort_values(by=['deadline_run'])

        df_cubic = data[data["cca"] == "cubic"]
        df_cubic = df_cubic[df_cubic["kernel"] == "kernel6-1"]
        df_cubic = df_cubic[["mbps","slice_perc","delay_rtt","rate"]].groupby(["slice_perc", "delay_rtt", "rate"],as_index=False).median()
        
        for rate_ in args["rate"]:
            if rate_ == 100:
                marker = "o"
            sns.stripplot(ax=axs[it], data=df_cubic[df_cubic["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", legend=False, jitter=False, alpha=0.9, palette=paletti, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)

        data_sorted=data[data["cca"] == bbr_version]
        for rate_ in args["rate"]:
            if rate_ == 100:
                marker = "o"
            sns.stripplot(ax=axs[it], data=data_sorted[data_sorted["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", jitter=0.25, alpha=0.3, palette=paletti, zorder=0, dodge=True, legend=True, marker = marker, size=3)
            sns.boxplot(ax=axs[it], data=data_sorted[data_sorted["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", palette=paletti,fliersize=0,legend=False)

        axs[it].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11,12], ["10","","20","","30","","40","","50","","60","","70"], fontsize=plotconfig.FONT_SIZE-3)
        axs[it].tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-3)
        axs[it].vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5,11.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)
        axs[it].set_xlim(-0.5, 12.5)
    
        axs[it].set_title(f"Timeslice {int(runti/1000000)}ms",fontsize=plotconfig.FONT_SIZE-3,pad=3)
        if runti == 1000000:
            axs[it].set_xlabel("")
            leg1=axs[it].legend(handles = [
                Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.1, marker="o", markersize=3, label='BBRv1' if bbr_version == "bbr" else "BBRv3"),
                Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.6, marker="o", markersize=1.8, label='Cubic'),
            ],loc="upper left", framealpha=0.5,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, bbox_to_anchor=(0, 0.85))
            axs[it].add_artist(leg1)
            labels = []
            h, l = axs[it].get_legend_handles_labels()
            for lii, hii in enumerate(h):
                hii.set_alpha(1)
                labels.append(f"{int(float(l[lii]))}ms")
            axs[it].legend(labels = labels, handles = h, loc="lower right", title="RTT", fontsize=plotconfig.FONT_SIZE-3, title_fontsize=plotconfig.FONT_SIZE-3, framealpha=0.9)
        elif runti == 10000000:
            axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)
            axs[it].legend([])
        elif runti == 20000000:
            axs[it].set_xlabel("")
            axs[it].legend([])

    axs[0].set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-3)
    axs[0].set_ylim(-float(args["rate"][0])*0.05,int(args["rate"][0])+5)
    
    if savefig:
        fig.savefig(f"figures/figure_{'8' if bbr_version == 'bbr3' else '15'}.pdf", format="pdf")
    else:
        plt.show()

<>:53: SyntaxWarning: invalid escape sequence '\%'
<>:53: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_240860/2849851258.py:53: SyntaxWarning: invalid escape sequence '\%'
  axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)


In [24]:
strip(df, True)

/tmp/ipykernel_240860/2849851258.py:24: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.stripplot(ax=axs[it], data=df_cubic[df_cubic["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", legend=False, jitter=False, alpha=0.9, palette=paletti, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
/tmp/ipykernel_240860/2849851258.py:30: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.stripplot(ax=axs[it], data=data_sorted[data_sorted["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", jitter=0.25, alpha=0.3, palette=paletti, zorder=0, dodge=True, legend=True, marker = marker, size=3)
/tmp/ipykernel_240860/2849851258.py:31: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.boxplot(ax=axs[it], data=data_sorted[data_sorted["rate"] == rate_], x="slice_perc", y="mbps", hue="delay_rtt", palette=paletti,fliersize=0,legen